# EEHEMTModelHelper Demo
This notebook demonstrates how to:
1. Create an EEHEMT model helper instance
2. Load init parameters from JSON
3. Simulate and plot I-V curves, then save figures

In [1]:
import importlib
import sys
from pathlib import Path

import numpy as np

notebook_dir = Path.cwd()
if not (notebook_dir / "eehemt_model_helper.py").exists():
    notebook_dir = notebook_dir / "test" / "eehemt_helper"

if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import eehemt_model_helper

importlib.reload(eehemt_model_helper)
from eehemt_model_helper import EEHEMTModelHelper

In [2]:
project_root = notebook_dir.parents[0]
va_path = notebook_dir / "eehemt114_2.va"
json_path = project_root / "modelcard_and_range/modelcard.json"
save_dir = project_root / "target_curve/"

helper = EEHEMTModelHelper(
    va_file_path=str(va_path),
    save_dir=str(save_dir),
    temperature=300,
    curve_condition_name="UGW",
)

print("Model created successfully.")

Model created successfully.


In [3]:
# Define gate-voltage sweep points
vgs = np.round(np.arange(0.0, 1.201, 0.025), 3)

vgs[:5], vgs[-5:]

(array([0.   , 0.025, 0.05 , 0.075, 0.1  ]),
 array([1.1  , 1.125, 1.15 , 1.175, 1.2  ]))

In [4]:
init_params = helper.load_init_params_from_json(str(json_path))
print("Init params loaded from JSON:")
for k, v in init_params.items():
    print(f"{k:>12s} = {v:.6g}")


Init params loaded from JSON:
         Vto = 0.335
         Vgo = 0.556
        Vsat = 0.456
         Vbc = 1.33
       Gmmax = 0.185
          Rd = 1.15
          Rs = 1.9
        Peff = 1.915
        Kapa = 0.059
       Alpha = 0.007


In [5]:
i_sim = helper.simulate_ids_from_json_init(
    json_path=str(json_path),
    vgs=vgs,
    vds_voltage=0.5,
    curve_condition_value=0.5,
    modelcard_updates=init_params,
)

i_sim[:5]

array([1.42807790e-29, 3.91751197e-28, 1.07465426e-26, 2.94799809e-25,
       8.08696626e-24])

In [6]:
linear_path = helper.plot_iv_curve(
    vgs=vgs,
    i_sim=i_sim,
    i_meas=None,
    title="EEHEMT Target I-V Curve (Linear)",
    save_name="iv_curve_linear_demo.png",
    log_y=False,
)

log_path = helper.plot_iv_curve(
    vgs=vgs,
    i_sim=i_sim,
    i_meas=None,
    title="EEHEMT Target I-V Curve (Log)",
    save_name="iv_curve_log_demo.png",
    log_y=True,
)

print("Linear plot:", linear_path)
print("Log plot:", log_path)

Saved plot to: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_linear_demo.png
Saved plot to: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_log_demo.png
Linear plot: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_linear_demo.png
Log plot: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_log_demo.png


# New Features: Update and Save Parameters

In [7]:
# Demonstrate update_modelcard()
# Modify some parameters programmatically
params_to_update = {
    "Vto": 0.4,
    "Vgo": 0.6,
    "Gmmax": 0.2,
}

helper.update_modelcard(params_to_update)
print("\nParameters after update:")
helper.print_params(list(params_to_update.keys()))


Updated 3 parameter(s) in modelcard.

Parameters after update:
         Vto = 0.4
         Vgo = 0.6
       Gmmax = 0.2


In [8]:
# Re-simulate and re-plot curve after parameter update
i_sim_updated = helper._simulate_ids(
    vgs=vgs,
    vds_voltage=0.5,
    curve_condition_value=0.5,
)

linear_updated_path = helper.plot_iv_curve(
    vgs=vgs,
    i_sim=i_sim_updated,
    i_meas=None,
    title="EEHEMT Target I-V Curve After Update (Linear)",
    save_name="iv_curve_linear_after_update_demo.png",
    log_y=False,
)

log_updated_path = helper.plot_iv_curve(
    vgs=vgs,
    i_sim=i_sim_updated,
    i_meas=None,
    title="EEHEMT Target I-V Curve After Update (Log)",
    save_name="iv_curve_log_after_update_demo.png",
    log_y=True,
)

print("Updated linear plot:", linear_updated_path)
print("Updated log plot:", log_updated_path)


Saved plot to: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_linear_after_update_demo.png
Saved plot to: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_log_after_update_demo.png
Updated linear plot: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_linear_after_update_demo.png
Updated log plot: /home/u5977862/DRL-on-parameter-extraction/demo/target_curve/iv_curve_log_after_update_demo.png


In [9]:
# Demonstrate save_params_to_json()
# Save updated parameters to a new JSON file in the same folder as reference modelcard.json
save_output_path = json_path.with_name("modelcard_updated_demo.json")

saved_path = helper.save_params_to_json(
    output_path=str(save_output_path), reference_json_path=str(json_path)
)

print(f"\nSaved parameters to: {saved_path}")


Saved 10 parameter(s) to: /home/u5977862/DRL-on-parameter-extraction/demo/modelcard_and_range/modelcard_updated_demo.json

Saved parameters to: /home/u5977862/DRL-on-parameter-extraction/demo/modelcard_and_range/modelcard_updated_demo.json


In [10]:
# Verify saved JSON file
import json

with open(save_output_path, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

print("Saved JSON file (first 3 parameters shown):")
for i, (name, cfg) in enumerate(saved_config.items()):
    if i >= 3:
        break
    print(f"{name}: {cfg}")

# Also test reloading from the saved file
helper2 = EEHEMTModelHelper(
    va_file_path=str(va_path),
    save_dir=str(save_dir),
    temperature=300,
    curve_condition_name="UGW",
)

reloaded_params = helper2.load_init_params_from_json(str(save_output_path))
print("\nReloaded parameters (first 3 shown):")
for i, (name, value) in enumerate(reloaded_params.items()):
    if i >= 3:
        break
    print(f"{name} = {value:.6g}")


Saved JSON file (first 3 parameters shown):
Vto: {'min': -1.0, 'max': 1.5, 'init': 0.4}
Vgo: {'min': 0.0, 'max': 1.5, 'init': 0.6}
Vsat: {'min': 0.1, 'max': 2.0, 'init': 0.456}

Reloaded parameters (first 3 shown):
Vto = 0.4
Vgo = 0.6
Vsat = 0.456
